# 01 - Data Preparation

**Purpose:** Load, explore, and preprocess microgrid input data (prices, solar, load).

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config.settings import load_config
from src.data.loaders import generate_synthetic_data

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Load Configuration

In [ ]:
settings = load_config('../config/config.yaml')
print(f"Region: {settings.region}")
print(f"Date range: {settings.start_date} to {settings.end_date}")
print(f"Resolution: {settings.resolution_minutes} minutes")

## Generate Synthetic Data

In [ ]:
df = generate_synthetic_data(
    start_date=settings.start_date,
    end_date=settings.end_date,
    resolution_minutes=settings.resolution_minutes,
    solar_capacity_kw=settings.solar_capacity_kw,
    timezone=settings.timezone,
)
df.head(10)

## Data Statistics

In [ ]:
df.describe()

## Visualize Daily Profiles

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Select first 24 hours
day_data = df.iloc[:24]

axes[0].plot(day_data.index, day_data['price'], 'b-', linewidth=2)
axes[0].set_ylabel('Price ($/kWh)')
axes[0].set_title('Electricity Spot Price')

axes[1].fill_between(day_data.index, day_data['solar'], alpha=0.7, color='gold')
axes[1].set_ylabel('Power (kW)')
axes[1].set_title('Solar Generation')

axes[2].plot(day_data.index, day_data['load'], 'k-', linewidth=2)
axes[2].set_ylabel('Power (kW)')
axes[2].set_title('Load Demand')
axes[2].set_xlabel('Time')

plt.tight_layout()
plt.show()

## Net Load Analysis

In [ ]:
df['net_load'] = df['load'] - df['solar']

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(df.index[:24], df['net_load'].iloc[:24], 
                where=df['net_load'].iloc[:24] > 0, alpha=0.7, color='red', label='Import')
ax.fill_between(df.index[:24], df['net_load'].iloc[:24], 
                where=df['net_load'].iloc[:24] < 0, alpha=0.7, color='green', label='Export')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Net Load (kW)')
ax.set_xlabel('Time')
ax.set_title('Net Load Profile (positive = import, negative = export)')
ax.legend()
plt.tight_layout()
plt.show()

## Save Processed Data

In [ ]:
from src.data.loaders import save_processed_data
from pathlib import Path

output_path = Path('../data/processed/microgrid_data.csv')
save_processed_data(df[['price', 'solar', 'load']], output_path)
print(f"Data saved to {output_path}")